### Morpheus

Morpheus was born from the need for a lightweight, high-performance tensor library that puts control and optimization back in the hands of researchers and developers — initially driven by a personal need to build a complete BSSN solver from scratch. Designed to be both modular and powerful, **Morpheus** emphasizes aligned memory, SIMD acceleration (SSE, AVX2, AVX512), multithreading, and optional MPI support — all without sacrificing readability or flexibility.  

The motivation behind **Morpheus** stems from a simple but demanding observation: many critical applications — ranging from simulations of spacetime geometry to machine learning training loops — require the repeated manipulation of large, dense vectors, matrices, and tensors. These operations must be performed as efficiently as possible on modern hardware, ideally without sacrificing clarity or interoperability.
s

Here is how to use the morpheus lib with python (search for the path because it was not used with pip and installed in the correct path):

In [54]:
import sys
import time 
import os
sys.path.append(os.path.abspath("pybuild"))
from morpheus import *

### Linear Algebra Recap

Linear algebra forms the backbone of most numerical simulations and machine learning algorithms. In this section, we briefly recall essential concepts such as vector spaces, matrix operations, inner products, and linear transformations — all of which serve as the mathematical foundation for the core functionalities of **Morpheus**.

#### Vectors and Matrices

Vectors and matrices are the fundamental objects of linear algebra. They represent data structures for encoding points in space, transformations, and systems of equations. **Morpheus** builds its core around efficient representations and operations on these objects, with a focus on performance and alignment.  

A matrix, in its most intrinsic mathematical sense, is a two-dimensional array of elements arranged in rows and columns. More formally, an $m \times n$ matrix is a function that assigns to each pair of indices $(i, j)$ a scalar entry $a_{ij}$, typically over a field such as $\mathbb{R}$ or $\mathbb{C}$. It can be seen as an element of the vector space $\mathbb{K}^{m \times n}$, where $\mathbb{K}$ denotes the base field.

In the special case where a matrix has only a single column (or row), it reduces to a vector, which we interpret as a point or direction in space. Vectors are thus a specific type of matrix, typically of size $n \times 1$ (column vector) or $1 \times n$ (row vector), and they inherit all structural and algebraic properties from the general matrix framework.  

Vectors can be represented either as column matrices in $\mathbb{K}^{n \times 1}$ or as row matrices in $\mathbb{K}^{1 \times n}$.


#### Vector Operations

The set $\mathbb{R}^n$ of all real $n$-dimensional vectors, equipped with the standard vector addition defined by  
$$(\mathbf{u} + \mathbf{v})_i := u_i + v_i \quad \text{for all } 1 \leq i \leq n,$$  
forms an abelian group $(\mathbb{R}^n, +)$, since addition is associative, commutative, has a neutral element (the zero vector), and each element has an additive inverse.

As an example, let  
$$
\mathbf{u} = \begin{pmatrix}
u_1 \\
u_2 \\
\vdots \\
u_n
\end{pmatrix}, \quad
\mathbf{v} = \begin{pmatrix}
v_1 \\
v_2 \\
\vdots \\
v_n
\end{pmatrix}
$$  
be two vectors in $\mathbb{R}^n$. Then their sum $\mathbf{w} = \mathbf{u} + \mathbf{v} \in \mathbb{R}^n$ is given by:
$$
\mathbf{w} = \begin{pmatrix}
u_1 + v_1 \\
u_2 + v_2 \\
\vdots \\
u_n + v_n
\end{pmatrix}.
$$

Just like vector addition, vector subtraction is defined componentwise. Given two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$, their difference $\mathbf{w} = \mathbf{u} - \mathbf{v} \in \mathbb{R}^n$ is defined by:  
$$(\mathbf{u} - \mathbf{v})_i := u_i - v_i \quad \text{for all } 1 \leq i \leq n.$$

This operation is well-defined and satisfies properties similar to addition, such as associativity with scalar multiplication and the existence of an additive inverse.  
However, **vector subtraction is not commutative in general**:
$$
\mathbf{u} - \mathbf{v} \neq \mathbf{v} - \mathbf{u} \quad \text{unless } \mathbf{u} = \mathbf{v}.
$$

For example, the difference of two vectors is:
$$
\mathbf{w} = \mathbf{u} - \mathbf{v} = \begin{pmatrix}
u_1 - v_1 \\
u_2 - v_2 \\
\vdots \\
u_n - v_n
\end{pmatrix}.
$$

The **dot product** (or inner product) of two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ is defined by:
$$
\langle \mathbf{u}, \mathbf{v} \rangle := \sum_{i=1}^n u_i v_i.
$$

This operation is bilinear, symmetric, and positive-definite. It induces the standard Euclidean norm:
$$
\| \mathbf{u} \| = \sqrt{ \langle \mathbf{u}, \mathbf{u} \rangle } = \sqrt{ \sum_{i=1}^n u_i^2 }.
$$


Scalar multiplication — or the **scaling of a vector by a scalar** — modifies both the magnitude and, possibly, the direction of the vector.

The operation **scl** denotes the scaling of a vector by a scalar, formally defined as the action of the field $\mathbb{K}$ on the vector space $\mathbb{K}^n$, mapping $(\lambda, \mathbf{v}) \mapsto \lambda \mathbf{v}$ such that each component of $\mathbf{v}$ is multiplied by $\lambda$.


In [55]:
print("\n add :\n")

v1 = Vector([1.0, 2.0, 3.0])
v2 = Vector([4.0, 5.0, 6.0])

print("v1 = ", v1)
print("v2 = ", v2)
print("v1 + v2 =", morph.add_vec(v1, v2))

print("\n sub :\n")

v3 = v1
v4 = v2
print("v3 = ", v3)
print("v4 = ", v4)
print("v3 - v4 =", morph.sub_vec(v3, v4))

print("\n scale (scl) :\n")
print("v1 scl 2", morph.scl_vec(v1, 2))




 add :

v1 =  [1, 2, 3]
v2 =  [4, 5, 6]
v1 + v2 = [5, 7, 9]

 sub :

v3 =  [1, 2, 3]
v4 =  [4, 5, 6]
v3 - v4 = [-3, -3, -3]

 scale (scl) :

v1 scl 2 [2, 4, 6]


#### Matrix Operations

The set $\mathbb{R}^{m \times n}$ of all real $m \times n$ matrices, equipped with the standard matrix addition defined by  
$$(A + B)_{ij} := A_{ij} + B_{ij} \quad \text{for all } 1 \leq i \leq m,\, 1 \leq j \leq n,$$  
forms an abelian group $(\mathbb{R}^{m \times n}, +)$, since addition is associative, commutative, has a neutral element (the zero matrix), and each element has an additive inverse.

As an example, we can therefore express the addition of matrices as:

Let  
$$A_{ij} = \begin{pmatrix}
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} & a_{m2} & \cdots & a_{mn}
\end{pmatrix}, \quad
B_{ij} = \begin{pmatrix}
b_{11} & b_{12} & \cdots & b_{1n} \\
b_{21} & b_{22} & \cdots & b_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
b_{m1} & b_{m2} & \cdots & b_{mn}
\end{pmatrix}$$  
be two matrices in $\mathbb{R}^{m \times n}$. Then their sum $C = A + B \in \mathbb{R}^{m \times n}$ is given by:  
$$
C_{ij} = \begin{pmatrix}
a_{11} + b_{11} & a_{12} + b_{12} & \cdots & a_{1n} + b_{1n} \\
a_{21} + b_{21} & a_{22} + b_{22} & \cdots & a_{2n} + b_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} + b_{m1} & a_{m2} + b_{m2} & \cdots & a_{mn} + b_{mn}
\end{pmatrix}.
$$

Just like matrix addition, matrix subtraction is defined componentwise. Given two matrices $A, B \in \mathbb{R}^{m \times n}$, their difference $C = A - B \in \mathbb{R}^{m \times n}$ is defined by:  
$$(A - B)_{ij} := A_{ij} - B_{ij} \quad \text{for all } 1 \leq i \leq m,\, 1 \leq j \leq n.$$

This operation is well-defined and satisfies properties similar to addition, such as associativity with respect to scalar multiplication and the existence of an additive inverse. However, **matrix subtraction is not commutative in general**:  
$$
A - B \neq B - A \quad \text{unless } A = B.
$$

For example, the difference of two matrices is given by:  
$$
C_{ij} = A_{ij} - B_{ij} = \begin{pmatrix}
a_{11} - b_{11} & a_{12} - b_{12} & \cdots & a_{1n} - b_{1n} \\
a_{21} - b_{21} & a_{22} - b_{22} & \cdots & a_{2n} - b_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} - b_{m1} & a_{m2} - b_{m2} & \cdots & a_{mn} - b_{mn}
\end{pmatrix}.
$$




In [56]:
A = Matrix(3, 3)
A.fill([
    [1.0, 2.0, 3.0],
    [0.0, 1.0, 4.0],
    [5.0, 6.0, 0.0],
])

v = Vector([1.0, 2.0, 3.0])


print("\nAddition A + A:")
print(morph.add_mat(A, A))

print("\nSubtraction A - A:")
print(morph.sub_mat(A, A))

print("\nScaling A * 2:")
print(morph.scl_mat(A, 2.0))


Addition A + A:
[
  [2, 4, 6],
  [0, 2, 8],
  [10, 12, 0]
]

Subtraction A - A:
[
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0]
]

Scaling A * 2:
[
  [2, 4, 6],
  [0, 2, 8],
  [10, 12, 0]
]


#### Linear Combinations

Let $\mathbb{K}$ be a field (typically $\mathbb{R}$ or $\mathbb{C}$), and let $V$ be a vector space over $\mathbb{K}$.  
Given a finite set of vectors $\{\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_k\} \subset V$ and scalars $\lambda_1, \lambda_2, \dots, \lambda_k \in \mathbb{K}$, a **linear combination** of these vectors is an expression of the form:
$$
\mathbf{u} = \lambda_1 \mathbf{v}_1 + \lambda_2 \mathbf{v}_2 + \cdots + \lambda_k \mathbf{v}_k = \sum_{i=1}^{k} \lambda_i \mathbf{v}_i.
$$

The vector $\mathbf{u} \in V$ obtained this way lies in the **span** of $\{\mathbf{v}_1, \dots, \mathbf{v}_k\}$, denoted by:
$$
\text{Span}(\mathbf{v}_1, \dots, \mathbf{v}_k) := \left\{ \sum_{i=1}^k \lambda_i \mathbf{v}_i \,\middle|\, \lambda_i \in \mathbb{K} \right\}.
$$

Linear combinations are foundational in linear algebra, as they define:
- the **structure** of vector spaces,
- the concept of **linear dependence** (a set of vectors is linearly dependent if at least one is a linear combination of the others),
- and the construction of **bases**.

In computational contexts (e.g., numerical methods or machine learning), linear combinations frequently appear in the form of:
- updates to solution vectors,
- weighted sums,
- or transformations in lower-dimensional subspaces.

For example, if  
$$
\mathbf{v}_1 = \begin{pmatrix}1 \\ 2 \\ 3\end{pmatrix}, \quad \mathbf{v}_2 = \begin{pmatrix}-1 \\ 0 \\ 1\end{pmatrix}, \quad \lambda_1 = 2, \quad \lambda_2 = 3,
$$  
then the linear combination is:
$$
\mathbf{u} = 2 \mathbf{v}_1 + 3 \mathbf{v}_2 = \begin{pmatrix}2 \\ 4 \\ 6\end{pmatrix} + \begin{pmatrix}-3 \\ 0 \\ 3\end{pmatrix} = \begin{pmatrix}-1 \\ 4 \\ 9\end{pmatrix}.
$$


In [57]:
from morpheus import Vector, morph

v1 = Vector([1.0, 2.0, 3.0])
v2 = Vector([4.0, 5.0, 6.0])
v3 = Vector([-1.0, 0.0, 1.0])

vectors = [v1, v2, v3]
coeffs = [2.0, -1.0, 0.5]

result = morph.linear_comb(vectors, coeffs)

print("Result:", result)


Result: [-2.5, -1, 0.5]


#### Linear Interpolation (lerp)

Linear interpolation, commonly abbreviated as **lerp**, is a fundamental operation that constructs a point along the line segment between two vectors. Given two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{K}^n$ and a scalar parameter $t \in [0, 1]$, the linear interpolation between $\mathbf{u}$ and $\mathbf{v}$ is defined as:
$$
\text{lerp}(\mathbf{u}, \mathbf{v}; t) := (1 - t)\mathbf{u} + t\mathbf{v}.
$$

This operation yields a new vector that lies on the segment joining $\mathbf{u}$ and $\mathbf{v}$:
- If $t = 0$, then $\text{lerp}(\mathbf{u}, \mathbf{v}; 0) = \mathbf{u}$.
- If $t = 1$, then $\text{lerp}(\mathbf{u}, \mathbf{v}; 1) = \mathbf{v}$.
- If $0 < t < 1$, the result is a **convex combination** of $\mathbf{u}$ and $\mathbf{v}$.

Lerp is a special case of a **linear combination** where the weights are $(1 - t)$ and $t$, and they always sum to 1. It is widely used in:
- computer graphics and animation (for motion interpolation),
- numerical simulations (for blending values),
- and optimization algorithms (for parameter continuation).

Geometrically, the result $\mathbf{w} = \text{lerp}(\mathbf{u}, \mathbf{v}; t)$ is the unique point on the straight line segment between $\mathbf{u}$ and $\mathbf{v}$, at a relative position $t$ from $\mathbf{u}$ toward $\mathbf{v}$.


In [58]:
print("lerp(v1, v2, 0.5) =", morph.lerp(v1, v2, 0.5))


lerp(v1, v2, 0.5) = [2.5, 3.5, 4.5]


#### Dot Product

The **dot product** (also known as the inner product in $\mathbb{R}^n$) is a bilinear, symmetric operation that maps two vectors to a scalar. Given two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$, their dot product is defined as:
$$
\mathbf{u} \cdot \mathbf{v} := \sum_{i=1}^{n} u_i v_i.
$$

This operation satisfies the following key properties:
- **Commutativity**: $\mathbf{u} \cdot \mathbf{v} = \mathbf{v} \cdot \mathbf{u}$
- **Bilinearity**:
  $$
  (\lambda \mathbf{u} + \mu \mathbf{v}) \cdot \mathbf{w} = \lambda (\mathbf{u} \cdot \mathbf{w}) + \mu (\mathbf{v} \cdot \mathbf{w}), \quad \forall \lambda, \mu \in \mathbb{R}
  $$
- **Positive-definiteness**: $\mathbf{u} \cdot \mathbf{u} \geq 0$, and $\mathbf{u} \cdot \mathbf{u} = 0$ if and only if $\mathbf{u} = \mathbf{0}$

The dot product induces the **Euclidean norm**:
$$
\| \mathbf{u} \| := \sqrt{\mathbf{u} \cdot \mathbf{u}} = \left( \sum_{i=1}^{n} u_i^2 \right)^{1/2}
$$

It also encodes geometric information, such as the **angle** $\theta$ between two vectors:
$$
\mathbf{u} \cdot \mathbf{v} = \| \mathbf{u} \| \, \| \mathbf{v} \| \cos \theta
$$
Thus:
- If $\mathbf{u} \cdot \mathbf{v} > 0$, the angle between them is acute.
- If $\mathbf{u} \cdot \mathbf{v} = 0$, the vectors are orthogonal.
- If $\mathbf{u} \cdot \mathbf{v} < 0$, the angle is obtuse.

The dot product is fundamental in linear algebra, geometry, and physics, and is central to algorithms in projection, optimization, and similarity computations.


In [59]:
print("\nDot product = \n")

print("V1 . V2 = ", morph.dot_vec(v1, v2))


Dot product = 

V1 . V2 =  32.0


#### Vector Norms

Given a vector $\mathbf{v} = (v_1, v_2, \dots, v_n) \in \mathbb{K}^n$ (with $\mathbb{K} = \mathbb{R}$ or $\mathbb{C}$), a **norm** is a function $\|\cdot\| : \mathbb{K}^n \rightarrow \mathbb{R}$ that satisfies:

- **Non-negativity**: $\|\mathbf{v}\| \geq 0$
- **Definiteness**: $\|\mathbf{v}\| = 0 \iff \mathbf{v} = \mathbf{0}$
- **Homogeneity**: $\|\lambda \mathbf{v}\| = |\lambda| \cdot \|\mathbf{v}\|$
- **Triangle inequality**: $\|\mathbf{u} + \mathbf{v}\| \leq \|\mathbf{u}\| + \|\mathbf{v}\|$

Below are the most commonly used norms:

---

**1. L¹ norm (Manhattan norm)**

Also called the **taxicab norm**, it is defined as the sum of the absolute values of the components:
$$
\|\mathbf{v}\|_1 := \sum_{i=1}^{n} |v_i|.
$$

Example:
$$
\mathbf{v} = \begin{pmatrix} 3 \\ -7 \\ 2 \end{pmatrix}, \quad \|\mathbf{v}\|_1 = |3| + |{-7}| + |2| = 12.
$$

---

**2. L² norm (Euclidean norm)**

This is the standard notion of length in Euclidean space, induced by the dot product:
$$
\|\mathbf{v}\|_2 := \sqrt{\sum_{i=1}^{n} |v_i|^2} = \sqrt{\mathbf{v} \cdot \mathbf{v}}.
$$

Example:
$$
\mathbf{v} = \begin{pmatrix} 3 \\ -7 \\ 2 \end{pmatrix}, \quad \|\mathbf{v}\|_2 = \sqrt{3^2 + (-7)^2 + 2^2} = \sqrt{62}.
$$

---

**3. L∞ norm (Infinity norm)**

Also called the **maximum norm**, it is the largest absolute component of the vector:
$$
\|\mathbf{v}\|_{\infty} := \max_{1 \leq i \leq n} |v_i|.
$$

Example:
$$
\mathbf{v} = \begin{pmatrix} 3 \\ -7 \\ 2 \end{pmatrix}, \quad \|\mathbf{v}\|_{\infty} = \max\{3, 7, 2\} = 7.
$$

---

These norms are all special cases of the **p-norm**:
$$
\|\mathbf{v}\|_p := \left( \sum_{i=1}^{n} |v_i|^p \right)^{1/p}, \quad p \geq 1.
$$
With:
- $p = 1 \Rightarrow$ L¹ norm
- $p = 2 \Rightarrow$ L² norm
- $p \to \infty \Rightarrow$ L∞ norm


In [60]:
print("norm_1 v1 = ", morph.norm_1(v1))
print("norm_2 v1 = ", morph.norm_2(v1))
print("norm_inf v1 = ", morph.norm_inf(v1))


norm_1 v1 =  6.0
norm_2 v1 =  3.7416574954986572
norm_inf v1 =  3.0


#### Cosine Similarity

The **cosine similarity** between two non-zero vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ is defined as the cosine of the angle $\theta$ between them:
$$
\cos(\theta) := \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \, \|\mathbf{v}\|_2}.
$$

This metric measures the **directional alignment** between $\mathbf{u}$ and $\mathbf{v}$, independently of their magnitudes.

**Properties:**
- $\cos(\theta) \in [-1, 1]$
- $\cos(\theta) = 1$ if $\mathbf{u}$ and $\mathbf{v}$ point in the **same direction**
- $\cos(\theta) = 0$ if $\mathbf{u}$ and $\mathbf{v}$ are **orthogonal**
- $\cos(\theta) = -1$ if they point in **opposite directions**

**Applications**:
- Frequently used in **machine learning**, **information retrieval**, and **natural language processing** to quantify similarity between high-dimensional vectors (e.g., document embeddings).
- In contrast to the Euclidean distance, cosine similarity is **scale-invariant**: multiplying either vector by a positive scalar does not affect the result.

**Example**:  
Let  
$$
\mathbf{u} = \begin{pmatrix}1 \\ 0\end{pmatrix}, \quad \mathbf{v} = \begin{pmatrix}1 \\ 1\end{pmatrix},
$$  
then
$$
\cos(\theta) = \frac{1 \cdot 1 + 0 \cdot 1}{\sqrt{1^2 + 0^2} \cdot \sqrt{1^2 + 1^2}} = \frac{1}{\sqrt{2}} \approx 0.707.
$$

This corresponds to an angle of $\theta = \frac{\pi}{4} = 45^\circ$.


In [61]:
u = Vector([1.0, 0.0])
v = Vector([1.0, 1.0])

cos_theta = morph.cosine(u, v)

print("u =", u)
print("v =", v)
print("cosine(u, v) =", cos_theta)


u = [1, 0]
v = [1, 1]
cosine(u, v) = 0.7071067690849304


#### Cross Product (3D Only)

The **cross product** is a binary operation defined only in $\mathbb{R}^3$, which takes two vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^3$ and returns a third vector $\mathbf{w} = \mathbf{u} \times \mathbf{v} \in \mathbb{R}^3$ that is orthogonal to both $\mathbf{u}$ and $\mathbf{v}$.

**Definition**:  
Let  
$$
\mathbf{u} = \begin{pmatrix} u_1 \\ u_2 \\ u_3 \end{pmatrix}, \quad
\mathbf{v} = \begin{pmatrix} v_1 \\ v_2 \\ v_3 \end{pmatrix},
$$  
then the cross product is defined as:
$$
\mathbf{u} \times \mathbf{v} = \begin{pmatrix}
u_2 v_3 - u_3 v_2 \\
u_3 v_1 - u_1 v_3 \\
u_1 v_2 - u_2 v_1
\end{pmatrix}.
$$

**Properties**:
- $\mathbf{u} \times \mathbf{v}$ is orthogonal to both $\mathbf{u}$ and $\mathbf{v}$
- $\|\mathbf{u} \times \mathbf{v}\| = \|\mathbf{u}\| \cdot \|\mathbf{v}\| \cdot \sin(\theta)$, where $\theta$ is the angle between $\mathbf{u}$ and $\mathbf{v}$
- **Anti-symmetric**: $\mathbf{u} \times \mathbf{v} = -(\mathbf{v} \times \mathbf{u})$
- $\mathbf{u} \times \mathbf{u} = \mathbf{0}$

Geometrically, the cross product gives a vector perpendicular to the plane defined by $\mathbf{u}$ and $\mathbf{v}$, with orientation determined by the **right-hand rule**.

**Example**:  
Let  
$$
\mathbf{u} = \begin{pmatrix}1 \\ 0 \\ 0\end{pmatrix}, \quad
\mathbf{v} = \begin{pmatrix}0 \\ 1 \\ 0\end{pmatrix},
$$  
then  
$$
\mathbf{u} \times \mathbf{v} = \begin{pmatrix}0 \\ 0 \\ 1\end{pmatrix}.
$$


In [62]:
u = Vector([1.0, 0.0, 0.0])
v = Vector([0.0, 1.0, 0.0])
w = morph.cross(u, v)

print("u =", u)
print("v =", v)
print("u × v =", w)


u = [1, 0, 0]
v = [0, 1, 0]
u × v = [0, 0, 1]


### Matrix product

The product of two matrices is, in general, not commutative; that is, $AB \neq BA$ in general.  
However, matrix multiplication is associative and distributive over matrix addition, by virtue of the algebraic rules governing this operation.

For a matrix product, let $A \in \mathbb{R}^{m \times p}$ and $B \in \mathbb{R}^{p \times n}$.  
The product $C = AB \in \mathbb{R}^{m \times n}$ is defined by:  
$$
C_{ij} := \sum_{k=1}^{p} A_{ik} B_{kj}, \quad \text{for all } 1 \leq i \leq m,\, 1 \leq j \leq n.
$$

This definition ensures that the matrix product is well-defined only when the number of columns of $A$ equals the number of rows of $B$. The resulting matrix $C$ has the same number of rows as $A$ and the same number of columns as $B$.

Let $A \in \mathbb{R}^{m \times p}$ and $B \in \mathbb{R}^{p \times n}$ be two matrices, defined as:

$$
A = \begin{pmatrix}
a_{11} & a_{12} & \cdots & a_{1p} \\
a_{21} & a_{22} & \cdots & a_{2p} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} & a_{m2} & \cdots & a_{mp}
\end{pmatrix}, \quad
B = \begin{pmatrix}
b_{11} & b_{12} & \cdots & b_{1n} \\
b_{21} & b_{22} & \cdots & b_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
b_{p1} & b_{p2} & \cdots & b_{pn}
\end{pmatrix}.
$$

Then their product $C = AB \in \mathbb{R}^{m \times n}$ is the matrix:

$$
C = \begin{pmatrix}
\sum_{k=1}^{p} a_{1k} b_{k1} & \sum_{k=1}^{p} a_{1k} b_{k2} & \cdots & \sum_{k=1}^{p} a_{1k} b_{kn} \\
\sum_{k=1}^{p} a_{2k} b_{k1} & \sum_{k=1}^{p} a_{2k} b_{k2} & \cdots & \sum_{k=1}^{p} a_{2k} b_{kdn} \\
\vdots & \vdots & \ddots & \vdots \\
\sum_{k=1}^{p} a_{mk} b_{k1} & \sum_{k=1}^{p} a_{mk} b_{k2} & \cdots & \sum_{k=1}^{p} a_{mk} b_{kn}
\end{pmatrix}.
$$

Matrix multiplication is **associative** and **distributive** over addition:  
$$
A(BC) = (AB)C, \quad A(B + C) = AB + AC, \quad (A + B)C = AC + BC.
$$

**Linearity with respect to rows and columns.**  
Let $A = (a_{ik})$, $B = (b_{kj})$, and $C = AB$. Then each entry $C_{ij}$ is obtained as the dot product of the $i$-th row of $A$ and the $j$-th column of $B$:  
$$
C_{ij} = \text{row}_i(A) \cdot \text{col}_j(B).
$$

In [63]:
A = Matrix(3, 3)
B = Matrix(3, 3)

A.fill([
    [2.0, 1.0, -1.0],
    [-3.0, -1.0, 2.0],
    [-2.0, 1.0, 2.0],
])

B.fill([
    [10.0, 1.0, 1.0],
    [2.0, 10.0, 1.0],
    [2.0, 2.0, 10.0],
])

R = morph.mul(A, B)
print("A . B =", R)


R2 = morph.mul_vec(A, u1)
print("A . u1 =", R2)

A . B = [
  [20, 10, -7],
  [-28, -9, 16],
  [-14, 12, 19]
]
A . u1 = [2, -3, -2]


#### Trace of a Matrix

The **trace** of a square matrix is defined as the sum of its diagonal elements.  
Let $A \in \mathbb{K}^{n \times n}$ be a square matrix over a field $\mathbb{K}$ (typically $\mathbb{R}$ or $\mathbb{C}$), then the **trace** of $A$ is given by:
$$
\mathrm{Tr}(A) := \sum_{i=1}^{n} A_{ii}.
$$

That is, the trace extracts the sum of the entries along the main diagonal of the matrix.

**Properties**:
- $\mathrm{Tr}(A + B) = \mathrm{Tr}(A) + \mathrm{Tr}(B)$
- $\mathrm{Tr}(\lambda A) = \lambda \cdot \mathrm{Tr}(A)$ for any scalar $\lambda$
- $\mathrm{Tr}(A^T) = \mathrm{Tr}(A)$
- $\mathrm{Tr}(AB) = \mathrm{Tr}(BA)$ (cyclic property, valid when dimensions match)
- The trace is **linear** and **basis-independent**

**Example**:  
If  
$$
A = \begin{pmatrix}
1 & 2 & 3 \\
0 & 4 & 5 \\
0 & 0 & 6
\end{pmatrix}, \quad \mathrm{Tr}(A) = 1 + 4 + 6 = 11.
$$


In [64]:
A = Matrix(3, 3)
A.fill([
    [1.0, 2.0, 3.0],
    [0.0, 4.0, 5.0],
    [0.0, 0.0, 6.0]
])

print("Matrix A:\n", A)
print("Trace of A =", morph.trace_mat(A))

Matrix A:
 [
  [1, 2, 3],
  [0, 4, 5],
  [0, 0, 6]
]
Trace of A = [
  [11]
]


#### Transpose of a Matrix

The **transpose** of a matrix is an operation that flips a matrix over its diagonal, exchanging its rows with its columns.  
Let $A \in \mathbb{K}^{m \times n}$ be a matrix over a field $\mathbb{K}$, then its **transpose**, denoted $A^T$, is defined by:
$$
(A^T)_{ij} := A_{ji}, \quad \text{for all } 1 \leq i \leq n,\ 1 \leq j \leq m.
$$

That is, if
$$
A = \begin{pmatrix}
a_{11} & a_{12} & \cdots & a_{1n} \\
a_{21} & a_{22} & \cdots & a_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m1} & a_{m2} & \cdots & a_{mn}
\end{pmatrix},
$$  
then
$$
A^T = \begin{pmatrix}
a_{11} & a_{21} & \cdots & a_{m1} \\
a_{12} & a_{22} & \cdots & a_{m2} \\
\vdots & \vdots & \ddots & \vdots \\
a_{1n} & a_{2n} & \cdots & a_{mn}
\end{pmatrix}.
$$

---

**Properties**:
- $(A^T)^T = A$ (involution)
- $(A + B)^T = A^T + B^T$
- $(\lambda A)^T = \lambda A^T$ for any scalar $\lambda$
- $(AB)^T = B^T A^T$ (reverses the product order)
- $\mathrm{Tr}(A^T) = \mathrm{Tr}(A)$

The transpose plays a fundamental role in linear algebra, particularly in defining symmetric matrices, adjoints, and orthogonality.

---

**Example**:  
Let  
$$
A = \begin{pmatrix}
1 & 2 & 3 \\
4 & 5 & 6
\end{pmatrix}, \quad
A^T = \begin{pmatrix}
1 & 4 \\
2 & 5 \\
3 & 6
\end{pmatrix}.
$$


In [65]:
A = Matrix(2, 3)
A.fill([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

AT = morph.transpose_mat(A)

print("A =\n", A)
print("A^T =\n", AT)


A =
 [
  [1, 2, 3],
  [4, 5, 6]
]
A^T =
 [
  [1, 4],
  [2, 5],
  [3, 6]
]


#### (interlude) Solving Linear Systems: Jacobi Method

The **Jacobi method** is an iterative algorithm used to solve a system of linear equations of the form:
$$
A \mathbf{x} = \mathbf{b},
$$
where $A \in \mathbb{R}^{n \times n}$ is a square matrix, $\mathbf{x}, \mathbf{b} \in \mathbb{R}^n$ are vectors.

---

**Idea**:  
Decompose the matrix $A$ into:
- $D$ = diagonal part of $A$
- $R$ = remainder (strictly off-diagonal part): $R = A - D$

Then rewrite the system as:
$$
\mathbf{x} = D^{-1}(\mathbf{b} - R \mathbf{x}).
$$

This yields the **Jacobi iteration**:
$$
\mathbf{x}^{(k+1)} = D^{-1}(\mathbf{b} - R \mathbf{x}^{(k)}),
$$
or component-wise:
$$
x_i^{(k+1)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j \ne i} a_{ij} x_j^{(k)} \right), \quad \text{for } i = 1, \dots, n.
$$

---

**Convergence Criteria**:
- The method converges if $A$ is **strictly diagonally dominant**, i.e.,
$$
|a_{ii}| > \sum_{j \ne i} |a_{ij}| \quad \text{for all } i.
$$
- Alternatively, convergence is guaranteed if $A$ is **symmetric positive definite**.

---

**Example**:  
Let
$$
A = \begin{pmatrix}
4 & 1 & 2 \\
3 & 5 & 1 \\
1 & 1 & 3
\end{pmatrix}, \quad
\mathbf{b} = \begin{pmatrix} 4 \\ 7 \\ 3 \end{pmatrix}
$$  
Starting with $\mathbf{x}^{(0)} = \mathbf{0}$, we compute iteratively using:
$$
x_1^{(k+1)} = \frac{1}{4} \left( 4 - x_2^{(k)} - 2x_3^{(k)} \right) \\
x_2^{(k+1)} = \frac{1}{5} \left( 7 - 3x_1^{(k)} - x_3^{(k)} \right) \\
x_3^{(k+1)} = \frac{1}{3} \left( 3 - x_1^{(k)} - x_2^{(k)} \right)
$$

---

**Stopping Criterion**:  
The iteration stops when:
$$
\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| < \varepsilon,
$$  
for a given tolerance $\varepsilon > 0$.



In [66]:
A = Matrix(3, 3)
A.fill([
    [4.0, 1.0, 2.0],
    [3.0, 5.0, 1.0],
    [1.0, 1.0, 3.0]
])
b = Vector([4.0, 7.0, 3.0])

x = morph.jacobi_solve(A, b, tol=1e-6, max_iter=100)

print("Solution x =", x)


Solution x = [0.5, 1, 0.5]


#### Solving Linear Systems: Gaussian Elimination

**Gaussian elimination** is a direct method to solve a linear system of the form:
$$
A \mathbf{x} = \mathbf{b}, \quad \text{with } A \in \mathbb{R}^{n \times n},\ \mathbf{b} \in \mathbb{R}^n.
$$

It consists of two main steps:
1. **Forward elimination**: Transform the augmented matrix $[A \mid \mathbf{b}]$ into an upper triangular form $[U \mid \mathbf{c}]$.
2. **Back substitution**: Solve the triangular system $U \mathbf{x} = \mathbf{c}$ starting from the last row up to the first.

---

**Step 1 – Forward Elimination**  
Eliminate the coefficients below the diagonal by replacing row $i$ with:
$$
R_i \leftarrow R_i - \frac{a_{ij}}{a_{jj}} R_j, \quad \text{for } i > j.
$$

This operation zeroes out the lower-triangular entries to produce an upper-triangular matrix $U$.

---

**Step 2 – Back Substitution**  
Once the matrix is in the form:
$$
U = \begin{pmatrix}
u_{11} & u_{12} & \cdots & u_{1n} \\
0 & u_{22} & \cdots & u_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & u_{nn}
\end{pmatrix}, \quad
\mathbf{c} = \begin{pmatrix} c_1 \\ c_2 \\ \vdots \\ c_n \end{pmatrix},
$$  
solve the system starting from:
$$
x_n = \frac{c_n}{u_{nn}}, \quad x_{n-1} = \frac{c_{n-1} - \sum_{j=n-1+1}^{n} u_{n-1,j} x_j}{u_{n-1,n-1}}, \quad \dots
$$

---

**Requirements**:
- Pivot elements $a_{ii} \ne 0$ (unless partial pivoting is used)
- Works for all non-singular square systems

**Example**:  
Solve:
$$
A = \begin{pmatrix}
2 & 1 & -1 \\
-3 & -1 & 2 \\
-2 & 1 & 2
\end{pmatrix}, \quad
\mathbf{b} = \begin{pmatrix}
8 \\
-11 \\
-3
\end{pmatrix}
$$

After forward elimination and back substitution, the solution is:
$$
\mathbf{x} = \begin{pmatrix} 2 \\ 3 \\ -1 \end{pmatrix}
$$


In [67]:
from morpheus import Matrix, Vector, morph

A = Matrix(3, 3)
A.fill([
    [2.0, 1.0, -1.0],
    [-3.0, -1.0, 2.0],
    [-2.0, 1.0, 2.0]
])
b = Vector([8.0, -11.0, -3.0])

x = morph.gauss_solve(A, b)

print("Solution x =", x)


Solution x = [2, 3, -1]


#### Trace of a Matrix

The **trace** of a square matrix is the sum of its diagonal elements.  
Given a matrix \( A \in \mathbb{K}^{n \times n} \), where \( \mathbb{K} \) is a field (typically \( \mathbb{R} \) or \( \mathbb{C} \)), the **trace** is defined by:
$$
\mathrm{Tr}(A) := \sum_{i=1}^{n} A_{ii}.
$$

That is, the trace extracts the sum of the elements on the main diagonal of \( A \).

---

**Properties**:

- **Linearity**:  
  $$
  \mathrm{Tr}(A + B) = \mathrm{Tr}(A) + \mathrm{Tr}(B), \quad \mathrm{Tr}(\lambda A) = \lambda \cdot \mathrm{Tr}(A)
  $$

- **Invariance under transpose**:  
  $$
  \mathrm{Tr}(A^T) = \mathrm{Tr}(A)
  $$

- **Cyclic property** (when dimensions allow):  
  $$
  \mathrm{Tr}(AB) = \mathrm{Tr}(BA)
  $$  
  but in general:  
  $$
  \mathrm{Tr}(ABC) \ne \mathrm{Tr}(CAB)
  $$

- **Similarity invariance**: If $B = P^{-1} A P$, then  
  $$
  \mathrm{Tr}(B) = \mathrm{Tr}(A)
  $$


---

**Example**:

Let  
$$
A = \begin{pmatrix}
3 & 2 & 1 \\
0 & 4 & 7 \\
5 & 6 & -2
\end{pmatrix}
$$  
Then:  
$$
\mathrm{Tr}(A) = 3 + 4 + (-2) = 5
$$


In [68]:
from morpheus import Matrix, morph

A = Matrix(3, 3)
A.fill([
    [3.0, 2.0, 1.0],
    [0.0, 4.0, 7.0],
    [5.0, 6.0, -2.0]
])

print("Matrix A:\n", A)
print("Trace of A =", morph.trace_mat(A))


Matrix A:
 [
  [3, 2, 1],
  [0, 4, 7],
  [5, 6, -2]
]
Trace of A = [
  [5]
]


#### Determinant of a Matrix

The **determinant** is a scalar value associated with a square matrix that encodes important information about the matrix, such as invertibility, volume scaling, and orientation.

Let \( A \in \mathbb{K}^{n \times n} \). The determinant of \( A \), denoted \( \det(A) \) or \( |A| \), is defined recursively by:

- For \( n = 1 \):  
  $$
  \det(A) = A_{11}
  $$

- For \( n \geq 2 \):  
  $$
  \det(A) = \sum_{j=1}^{n} (-1)^{1+j} A_{1j} \cdot \det(M_{1j})
  $$
  where \( M_{1j} \) is the \((n-1) \times (n-1)\) minor of \( A \) obtained by deleting row 1 and column \( j \).

---

**Example (3×3)**:  
Let  
$$
A = \begin{pmatrix}
a & b & c \\
d & e & f \\
g & h & i
\end{pmatrix}
$$  
Then:  
$$
\det(A) = aei + bfg + cdh - afh - bdi - ceg
$$

---

**Key Properties**:

- \( \det(I_n) = 1 \) (identity matrix)
- \( \det(A^T) = \det(A) \)
- \( \det(AB) = \det(A)\det(B) \)
- \( \det(\lambda A) = \lambda^n \det(A) \) for scalar \( \lambda \)
- \( A \text{ invertible } \iff \det(A) \ne 0 \)
- Row operations:
  - Swapping two rows changes the sign
  - Multiplying a row by \( \lambda \) multiplies the determinant by \( \lambda \)
  - Adding a multiple of one row to another does not change the determinant

---

**Geometric Interpretation**:
- In \( \mathbb{R}^2 \) or \( \mathbb{R}^3 \), the absolute value \( |\det(A)| \) gives the **volume scaling factor** of the linear transformation represented by \( A \).
- A negative sign indicates a **change of orientation**.

---

**Example**:  
Let  
$$
A = \begin{pmatrix}
1 & 2 \\
3 & 4
\end{pmatrix}
\quad \Rightarrow \quad
\det(A) = 1 \cdot 4 - 2 \cdot 3 = -2
$$


In [69]:
from morpheus import Matrix, morph

A = Matrix(2, 2)
A.fill([
    [1.0, 2.0],
    [3.0, 4.0]
])

print("Determinant of A =", morph.det_mat(A))

Determinant of A = -1.9999998807907104


#### Inverse of a Matrix

Let $$ A \in \mathbb{K}^{n \times n} $$ be a square matrix over a field $$ \mathbb{K} $$.  
The **inverse** of $$ A $$, denoted $$ A^{-1} $$, is defined as the unique matrix such that:

$$
A A^{-1} = A^{-1} A = I_n,
$$

where $$ I_n $$ is the identity matrix of size $$ n \times n $$.

---

**Existence condition**:  
A matrix is invertible (also called **non-singular**) if and only if:

$$
\det(A) \ne 0.
$$

---

**Formula for the inverse (via adjugate)**:

For $$ A \in \mathbb{R}^{n \times n} $$,

$$
A^{-1} = \frac{1}{\det(A)} \cdot \mathrm{adj}(A),
$$

where $$ \mathrm{adj}(A) $$ is the **adjugate** matrix, i.e., the transpose of the cofactor matrix of $$ A $$.

---

**Properties**:

$$
(A^{-1})^{-1} = A
$$

$$
(AB)^{-1} = B^{-1} A^{-1}
$$

$$
(A^T)^{-1} = (A^{-1})^T
$$

$$
\det(A^{-1}) = \frac{1}{\det(A)}
$$

---

**Example**:  
Let

$$
A = \begin{pmatrix}
4 & 7 \\
2 & 6
\end{pmatrix}, \quad
\det(A) = 4 \cdot 6 - 7 \cdot 2 = 24 - 14 = 10
$$

Then

$$
A^{-1} = \frac{1}{10} \begin{pmatrix}
6 & -7 \\
-2 & 4
\end{pmatrix}
= \begin{pmatrix}
0.6 & -0.7 \\
-0.2 & 0.4
\end{pmatrix}
$$


In [70]:
from morpheus import Matrix, morph

A = Matrix(2, 2)
A.fill([
    [4.0, 7.0],
    [2.0, 6.0]
])

A_inv = morph.inverse_mat(A)

print("A⁻¹ =\n", A_inv)


A⁻¹ =
 [
  [0.6, -0.7],
  [-0.2, 0.4]
]


#### Rank of a Matrix

The **rank** of a matrix is the dimension of the vector space spanned by its rows or its columns.

Let  
$$
A \in \mathbb{K}^{m \times n}
$$  
then the **rank** of $$A$$, denoted $$\mathrm{rank}(A)$$, is defined as:

$$
\mathrm{rank}(A) := \dim(\mathrm{Im}(A)) = \text{maximum number of linearly independent rows or columns}.
$$

---

**Equivalently**, the rank is:

- The dimension of the **column space** of $$A$$
- The dimension of the **row space** of $$A$$
- The number of non-zero rows in the **row echelon form** of $$A$$
- The size of the largest non-zero **minor** (determinant of a square submatrix)

---

**Properties**:

$$
\mathrm{rank}(A) \leq \min(m, n)
$$

$$
\mathrm{rank}(A) = \mathrm{rank}(A^T)
$$

If $$ A \in \mathbb{K}^{n \times n} $$:

$$
A \text{ is invertible} \iff \mathrm{rank}(A) = n
$$

For matrix multiplication:

$$
\mathrm{rank}(AB) \leq \min(\mathrm{rank}(A), \mathrm{rank}(B))
$$

---

**Geometric Interpretation**:

The rank of a matrix corresponds to the number of **linearly independent directions** represented by the matrix.  
It also equals the dimension of the image (or range) of the associated linear transformation.

---

**Example**:

Let  
$$
A = \begin{pmatrix}
1 & 2 & 3 \\
2 & 4 & 6 \\
0 & 0 & 1
\end{pmatrix}
$$

We observe that the second row is linearly dependent on the first:  
$$
\text{row}_2 = 2 \cdot \text{row}_1
$$  
but the third row is linearly independent.

Thus:  
$$
\mathrm{rank}(A) = 2
$$


In [ ]:
from morpheus import Matrix, morph

A = Matrix(3, 3)
A.fill([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [0.0, 0.0, 1.0]
])

print("Rank of A =", morph.rank_mat(A))


Rank of A = 2
